# OpenCode with Ollama in Colab



## Setup


In [ ]:
# @title Install packages
%%capture output
%%bash

# bash setup
fgrep -q '.bash_aliases' ~/.bashrc || {
  echo -e '\n\n\n[ -f ~/.bash_aliases ] && source ~/.bash_aliases\n' >> ~/.bashrc
}

{ cat << 'eof'
alias cls=clear
alias dir='ls -la'
eof
} > ~/.bash_aliases

{ cat << 'eof'

export PATH='/root/.local/bin':$PATH
eof
} >> ~/.bashrc


# system package installs
tmux new -s update -d " \
  apt-get update ;\
  apt-get install -y zstd ;\
  apt-get install -y tree jq ncal less texlive-xetex pandoc ; \
  echo == Done ; \
  sleep 30
"


# jupyter install
tmux new -s jupyter-server -d " \
  pip install ipyaml jupyterlab ; \
  jupyter labextension disable @jupyterlab/apputils-extension:announcements ; \
  jupyter lab \
    --ip=127.0.0.1 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token='' ; \
  echo == Done ; \
  sleep 30
"

# claude code install
while ! which zstd ; do sleep 1 ;done
curl -fsSL https://claude.ai/install.sh | bash
echo 'export PATH='/root/.local/bin':$PATH' >> ~/.bashrc


# opencode install
while ! which zstd ; do sleep 1 ;done
curl -fsSL https://opencode.ai/install | bash


# ollama service install and launch
tmux new -s ollama -d "\
  mkdir /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.log 2>&1 ; \
  while ! which zstd ; do sleep 1 ;done ; \
  curl -fsSL https://ollama.com/install.sh | sh ; \
  OLLAMA_KEEP_ALIVE=20m ollama serve ; \
  echo == Done ; \
  sleep 10
"


# ollama models pull and load
tmux new -s ollama_models -d "\
  mkdir /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.models.log 2>&1 ; \
  while ! curl -s -I 127.0.0.1:11434 ; do date ; sleep 1 ; done ;\
  ollama pull qwen2.5-coder:7b-instruct-q8_0 ; \
  ollama run qwen2.5-coder:7b-instruct-q8_0 --keepalive 20m "" ;\
  ollama ps ; \
  echo deepseek-coder-v2:16b-lite-instruct-q4_K_M qwen2.5-coder:14b-instruct-q4_K_M  | xargs -n 1 -P 5 ollama pull ; \
  echo ; \
  echo == Done ; \
  sleep 10
"



In [ ]:
# @title Modules, etc.
%alias tree tree

from datetime import datetime, timezone
from time import sleep
from google.colab import output

print("Click URL to open Jupyter Lab")
output.serve_kernel_port_as_window(8888)


When ready, click on the link to Jupyter Lab, open a terminal, and type this to start opencode:

```
ollama launch opencode
```

... and scroll down to select the local model.  Then wait less than a minute for the prompt appear.

or ... type this to interact with ollama chat:

```
ollama run qwen3:1.7b
```





In [ ]:
# @title Timer 1
# show a timer for 30 minutes
print("Timer 1")
for i in range(60*30):
  utc_now = datetime.now(timezone.utc)
  print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
  sleep(1)
